# Vežba 7: Stabla, ansambli i hiperparametri

Stablo odlučivanja, slučajna šuma i gradient boosting, unakrsna validacija, pretraga hiperparametara, važnost obeležja i pošteno poređenje modela

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba7.html](https://isatvss.surge.sh/vezba7.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Stablo odlučivanja

### Zadatak 1. Priprema podataka

Pripremiti skup nezgoda i podelu na trening i test kao na vežbama 5 i 6. Funkcija za pripremu sada ima parametar kojim se skaliranje može isključiti, jer stablima nije potrebno.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

nezgode = pd.read_csv("https://isatvss.surge.sh/podaci/nezgode_2025.csv")
nezgode = nezgode.drop_duplicates(subset="id_nezgode").reset_index(drop=True)
nezgode["teska"] = (nezgode["posledica"] != "Sa mat.stetom").astype(int)
vreme = pd.to_datetime(nezgode["datum_vreme"], format="%d.%m.%Y,%H:%M")
nezgode["sat"] = vreme.dt.hour
nezgode["dan_u_nedelji"] = vreme.dt.dayofweek
nezgode["mesec"] = vreme.dt.month

numericke = ["sat", "dan_u_nedelji", "mesec", "lon", "lat"]
kategoricke = ["policijska_uprava", "vrsta", "opis"]
X = nezgode[numericke + kategoricke]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, nezgode["teska"], test_size=0.2, stratify=nezgode["teska"], random_state=42
)


def one_hot_priprema(skaliranje=True):
    numericki_deo = [SimpleImputer(strategy="median")]
    if skaliranje:
        numericki_deo.append(StandardScaler())
    return ColumnTransformer(
        [
            ("num", make_pipeline(*numericki_deo), numericke),
            ("kat", OneHotEncoder(handle_unknown="ignore"), kategoricke),
        ]
    )


print(f"Trening: {len(X_tr)}, test: {len(X_te)} nezgoda")

### Zadatak 2. Stablo dubine 3

Istrenirati stablo najveće dubine 3 i nacrtati ga.

Prvo pitanje je da li je nezgoda sa pešakom. Ova grana, sa 7,7 % nezgoda, gotovo je čista: 98,6 % su teške. To je isti nalaz kao u deskriptivnoj analizi na vežbi 1, ali ovde ga je model pronašao sam. Ovako malo stablo ima tačnost 0,694 i lako se objašnjava, ali je slabije od logističke regresije.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

stablo_malo = Pipeline(
    [
        ("priprema", one_hot_priprema(skaliranje=False)),
        ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ]
)
stablo_malo.fit(X_tr, y_tr)
nazivi = [
    n.replace("kat__", "").replace("num__", "")
    for n in stablo_malo[0].get_feature_names_out()
]

plt.figure(figsize=(16, 6))
plot_tree(
    stablo_malo[-1],
    feature_names=nazivi,
    class_names=["laka", "teška"],
    filled=True,
    impurity=True,
    proportion=True,
    fontsize=8,
    rounded=True,
)
plt.show()
print(f"Tačnost stabla dubine 3 na testu: {stablo_malo.score(X_te, y_te):.3f}")

pesaci = X_tr["vrsta"] == "SN SA PEŠACIMA"
print(f"Udeo nezgoda sa pešacima u treningu: {pesaci.mean():.1%}")
print(f"Udeo teških među njima: {y_tr[pesaci].mean():.1%}")

### Zadatak 3. Dubina stabla i overfitting

Istrenirati stabla različite najveće dubine i nacrtati tačnost na treningu i na testu.

Tačnost na testu raste do dubine oko 12 (0,729), a zatim opada, dok tačnost na treningu stalno raste i na dubini 30 dostiže 0,982. Stablo dubine 30 pamti trening skup. Isti obrazac sa vežbe 4 pojavljuje se kod svakog modela čija se složenost može podešavati.

In [ ]:
dubine = [1, 2, 3, 5, 8, 12, 16, 20, 30]
tacnost_tr, tacnost_te = [], []
for dubina in dubine:
    stablo = Pipeline(
        [
            ("priprema", one_hot_priprema(skaliranje=False)),
            ("model", DecisionTreeClassifier(max_depth=dubina, random_state=42)),
        ]
    )
    stablo.fit(X_tr, y_tr)
    tacnost_tr.append(stablo.score(X_tr, y_tr))
    tacnost_te.append(stablo.score(X_te, y_te))

plt.figure(figsize=(7, 4))
plt.plot(dubine, tacnost_tr, "-o", color="#5b3cc4", label="trening")
plt.plot(dubine, tacnost_te, "-o", color="#d9480f", label="test")
plt.xlabel("Najveća dubina stabla")
plt.ylabel("Tačnost")
plt.legend()
plt.show()
print(
    pd.DataFrame({"dubina": dubine, "trening": tacnost_tr, "test": tacnost_te})
    .round(3)
    .to_string(index=False)
)

## Ansambli stabala

### Zadatak 4. Slučajna šuma

Istrenirati slučajnu šumu od 200 stabala i uporediti njen ROC AUC sa jednim stablom dubine 8.

Jedno stablo ima ROC AUC 0,807, a šuma 0,837. Usrednjavanje mnogo stabala smanjuje varijansu pojedinačnog stabla, pa model bolje radi na novim podacima.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

suma = Pipeline(
    [
        ("priprema", one_hot_priprema(skaliranje=False)),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42
            ),
        ),
    ]
)
suma.fit(X_tr, y_tr)
p_suma = suma.predict_proba(X_te)[:, 1]
stablo_8 = Pipeline(
    [
        ("priprema", one_hot_priprema(skaliranje=False)),
        ("model", DecisionTreeClassifier(max_depth=8, random_state=42)),
    ]
).fit(X_tr, y_tr)
p_stablo = stablo_8.predict_proba(X_te)[:, 1]
print(f"Jedno stablo (dubina 8): ROC AUC = {roc_auc_score(y_te, p_stablo):.3f}")
print(f"Slučajna šuma, 200 stabala: ROC AUC = {roc_auc_score(y_te, p_suma):.3f}")

### Zadatak 5. Gradient boosting sa kategoričkim obeležjima

Istrenirati `HistGradientBoostingClassifier` sa ugrađenom podrškom za kategorička obeležja i uporediti ROC AUC i AP tri modela: logističke regresije, slučajne šume i gradient boosting-a.

Ansambli su bolji od logističke regresije, ali malo: ROC AUC 0,837 i 0,836 naspram 0,826, a AP 0,785 i 0,781 naspram 0,771. Kada su obeležja uglavnom kategorička i jaka, kao vrsta i opis nezgode, i linearni model hvata najveći deo informacije.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

redosledna_priprema = ColumnTransformer(
    [
        ("num", "passthrough", numericke),
        (
            "kat",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            kategoricke,
        ),
    ]
)
kategoricke_kolone = [False] * len(numericke) + [True] * len(kategoricke)
boosting = Pipeline(
    [
        ("priprema", redosledna_priprema),
        (
            "model",
            HistGradientBoostingClassifier(
                categorical_features=kategoricke_kolone, random_state=42
            ),
        ),
    ]
)
boosting.fit(X_tr, y_tr)
p_boosting = boosting.predict_proba(X_te)[:, 1]

logisticka = Pipeline(
    [("priprema", one_hot_priprema()), ("model", LogisticRegression(max_iter=1000))]
).fit(X_tr, y_tr)
p_log = logisticka.predict_proba(X_te)[:, 1]
for naziv, p in [
    ("logistička regresija", p_log),
    ("slučajna šuma", p_suma),
    ("gradient boosting", p_boosting),
]:
    print(f"{naziv:22s} ROC AUC = {roc_auc_score(y_te, p):.3f}", end="")
    print(f", AP = {average_precision_score(y_te, p):.3f}")

## Unakrsna validacija i hiperparametri

### Zadatak 6. Poređenje modela unakrsnom validacijom

Za tri modela izračunati ROC AUC u petostrukoj stratifikovanoj unakrsnoj validaciji na trening skupu, sa prosekom i standardnom devijacijom.

Slučajna šuma ima prosek 0,844 ± 0,005, gradient boosting 0,842 ± 0,006, a logistička regresija 0,833 ± 0,006. Razlika između dva ansambla manja je od rasipanja između foldova, pa se ne može tvrditi da je jedan bolji od drugog. Logistička regresija je slabija u svakom foldu, pa je njeno zaostajanje stvarno.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

podela = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for naziv, m in [
    ("logistička regresija", logisticka),
    ("slučajna šuma", suma),
    ("gradient boosting", boosting),
]:
    rezultati = cross_val_score(m, X_tr, y_tr, cv=podela, scoring="roc_auc")
    print(f"{naziv:22s} AUC po foldovima: {np.round(rezultati, 3)}", end="")
    print(f", prosek {rezultati.mean():.3f} ± {rezultati.std():.3f}")

### Zadatak 7. Pretraga hiperparametara

Nasumično isprobati 12 kombinacija hiperparametara gradient boosting-a i za svaku izračunati prosečan ROC AUC u unakrsnoj validaciji. Ispisati najbolju kombinaciju i pet najboljih rezultata.

Najbolja kombinacija ima prosečan AUC 0,843, gotovo isto kao podrazumevana podešavanja. Pet najboljih kombinacija razlikuje se tek u četvrtoj decimali. Pretraga hiperparametara retko donosi velik pomak. Njena vrednost je u tome što pokazuje da rezultat nije posledica slučajno dobro ili loše izabranih podešavanja.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

prostor = {
    "model__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [10, 20, 50, 100],
    "model__l2_regularization": [0.0, 0.1, 1.0],
}
pretraga = RandomizedSearchCV(
    boosting, prostor, n_iter=12, cv=podela, scoring="roc_auc", random_state=42
)
pretraga.fit(X_tr, y_tr)
print("Najbolji hiperparametri:")
for ime, vrednost in pretraga.best_params_.items():
    print(f"  {ime.replace('model__', '')} = {vrednost}")
print(f"AUC unakrsne validacije: {pretraga.best_score_:.3f}")

rezultati = pd.DataFrame(pretraga.cv_results_)
kolone = [
    "param_model__learning_rate",
    "param_model__max_leaf_nodes",
    "mean_test_score",
]
print()
print(
    rezultati.sort_values("rank_test_score")[kolone]
    .head(5)
    .round(4)
    .to_string(index=False)
)

## Važnost obeležja i analiza grešaka

### Zadatak 8. Permutaciona važnost

Za izabrani model izmeriti koliko ROC AUC na testu pada kada se vrednosti jedne kolone nasumično ispremeštaju, za svaku kolonu posebno.

Najvažniji je opis nezgode (pad 0,1628), a slede policijska uprava (0,0623) i vrsta nezgode (0,0586). Vrsta ispada manje važna nego što je bila u deskriptivnoj analizi jer je opis detaljniji i sadrži istu informaciju. Kada se vrsta pomeša, model se osloni na opis. Permutaciona važnost tako potcenjuje obeležja koja su povezana sa drugim obeležjima, što se u izveštaju navodi uz rezultat.

In [ ]:
from sklearn.inspection import permutation_importance

najbolji = pretraga.best_estimator_
vaznost = permutation_importance(
    najbolji, X_te, y_te, scoring="roc_auc", n_repeats=5, random_state=42
)
tabela = pd.DataFrame(
    {"pad_AUC": vaznost.importances_mean, "std": vaznost.importances_std},
    index=X_te.columns,
).sort_values("pad_AUC", ascending=False)
print(tabela.round(4))

### Zadatak 9. Konačno poređenje i greške modela

Uporediti sve modele na testu i ispisati teške nezgode kojima je izabrani model dao najmanju verovatnoću.

Na testu podešen model ima ROC AUC 0,837, isto kao slučajna šuma, što potvrđuje zaključak unakrsne validacije. Teške nezgode kojima model daje najmanju verovatnoću, oko 0,019, uglavnom su nezgode sa jednim vozilom i preprekom na kolovozu. Takve nezgode su najčešće lake, a koje su teške, iz raspoloživih obeležja ne može se zaključiti. Za to bi bili potrebni podaci o brzini, putu ili vozilu, kojih u skupu nema.

In [ ]:
p_najbolji = najbolji.predict_proba(X_te)[:, 1]
verovatnoce = {
    "logistička regresija": p_log,
    "slučajna šuma": p_suma,
    "boosting": p_boosting,
    "boosting, podešen": p_najbolji,
}
poredjenje = pd.DataFrame(
    {
        "ROC AUC": {n: roc_auc_score(y_te, p) for n, p in verovatnoce.items()},
        "AP": {n: average_precision_score(y_te, p) for n, p in verovatnoce.items()},
    }
)
print(poredjenje.round(3))

greske = X_te.assign(stvarno=y_te.values, p=p_najbolji)
propustene = greske[(greske["stvarno"] == 1)].sort_values("p").head(5)
print("\nTeške nezgode kojima je model dao najmanju verovatnoću:")
print(propustene[["vrsta", "opis", "p"]].round(3).to_string(index=False))

## Zadaci za samostalan rad

### Zadatak 10. Gini ručno

Čvor sadrži 40 teških i 60 lakih nezgoda. Podela ga deli na dva čvora od po 50 nezgoda, sa 10 teških levo i 30 teških desno. Izračunati Gini nečistoću roditelja, oba deteta, ponderisanu nečistoću posle podele i smanjenje nečistoće.

### Zadatak 11. Broj stabala u šumi

Istrenirati slučajnu šumu sa 1, 10, 50 i 200 stabala i uporediti ROC AUC na testu. Posle koliko stabala dodavanje novih prestaje da pomaže?

### Zadatak 12. Gradient boosting za regresiju

Na podacima o vazduhu sa vežbe 4 uporediti `HistGradientBoostingRegressor` i Ridge regresiju, sa istim obeležjima i istom vremenskom podelom.

### Zadatak 13. Dve vrste važnosti

Iz slučajne šume pročitati važnost po smanjenju nečistoće, sabrati je po izvornoj koloni i uporediti redosled sa permutacionom važnošću iz zadatka 8. Koja obeležja dobijaju različito mesto i zašto?

### Zadatak 14. Rezultati pretrage

Iz `cv_results_` ispisati svih 12 isprobanih kombinacija sa prosekom, standardnom devijacijom i razmakom od najboljeg rezultata. Da li je razmak između najbolje i najlošije kombinacije veći od standardne devijacije?